In [1]:
%cd ../..

/home/eli/AnacondaProjects/epych


In [2]:
import collections
import glob
import functools
import logging
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import quantities as pq

import epych
from epych.statistics import alignment

[striatum:122281] shmem: mmap: an error occurred while determining whether or not /tmp/ompi.striatum.1000/jf.0/713097216/shared_mem_cuda_pool.striatum could be created.
[striatum:122281] create_and_attach: unable to create shared memory BTL coordinating structure :: size 134217728 


In [3]:
%matplotlib inline

In [4]:
logging.basicConfig(level=logging.INFO)

In [5]:
CONDITIONS = ["go_gloexp", "go_seqctl", "lo_gloexp", "lonaive", "lo_rndctl", "igo_seqctl"]
PRETRIAL_SECONDS = 0.5
POSTTRIAL_SECONDS = 0.5

In [6]:
NWB_SUBJECTS = glob.glob('/mnt/data/000253/sub-*/')

In [7]:
PILOT_FILES = []

In [8]:
NUM_TRIALS = 0
ODDBALL_ONSET = 0.
ODDBALL_OFFSET = 0.

In [9]:
aligner = epych.statistics.alignment.AlignmentSummary.unpickle("/mnt/data/000253/visual_alignment")

In [10]:
def visual_align(signal):
    return signal.select_channels(["VIS" in loc for loc in signal.channels.location]).median_filter()

In [11]:
def samplings(cond):
    for s, subject_dir in enumerate(sorted(NWB_SUBJECTS)):
        subject = subject_dir.split('/')[-2]
        if not os.path.exists(subject_dir + "/" + cond):
            continue
        sampling = epych.recording.Sampling.unpickle(subject_dir + "/" + cond).smap(visual_align)
        global ODDBALL_ONSET
        global ODDBALL_OFFSET
        global NUM_TRIALS
        ODDBALL_ONSET += sampling.trials['stim3_start'].sum()
        ODDBALL_OFFSET += sampling.trials['stim3_end'].sum()
        NUM_TRIALS += len(sampling.trials)
        yield sampling
        del sampling
        logging.info("Loaded LFPs for %s in subject %s" % (cond, subject))

In [12]:
for cond in CONDITIONS:
    samplings(cond)

In [13]:
def initialize_grandcat(key, signal):
    area = os.path.commonprefix([loc for loc in signal.channels.location])
    return epych.statistics.grand.GrandConcatenation(aligner.stats[area])

In [14]:
for cond in CONDITIONS:
    summary = epych.statistic.Summary(alignment.location_prefix, initialize_grandcat)
    summary.calculate(samplings(cond))
    cat = summary.results()
    del summary
    cat.pickle("/mnt/data/000253/grandcat_%s" % cond)
    logging.info("Grand-concatenated LFPs for condition %s" % cond)
    del cat

INFO:root:Loaded LFPs for go_gloexp in subject sub-621890
INFO:root:Loaded LFPs for go_gloexp in subject sub-632485
INFO:root:Loaded LFPs for go_gloexp in subject sub-632487
INFO:root:Loaded LFPs for go_gloexp in subject sub-637542
INFO:root:Loaded LFPs for go_gloexp in subject sub-637908
INFO:root:Loaded LFPs for go_gloexp in subject sub-637909
INFO:root:Loaded LFPs for go_gloexp in subject sub-640507
INFO:root:Loaded LFPs for go_gloexp in subject sub-642507
INFO:root:Loaded LFPs for go_gloexp in subject sub-645322
INFO:root:Loaded LFPs for go_gloexp in subject sub-645324
INFO:root:Loaded LFPs for go_gloexp in subject sub-645495
INFO:root:Loaded LFPs for go_gloexp in subject sub-647836
INFO:root:Loaded LFPs for go_gloexp in subject sub-649323
INFO:root:Loaded LFPs for go_gloexp in subject sub-649324
INFO:root:Grand-concatenated LFPs for condition go_gloexp
INFO:root:Loaded LFPs for go_seqctl in subject sub-621890
INFO:root:Loaded LFPs for go_seqctl in subject sub-632485
INFO:root:Load